[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-1-python-for-ai/milestone-assignment/solution/solution.ipynb)

# Module 1 milestone: reference solution

Loads a dataset, computes facts with Pandas, asks a local LLM to summarize them, and saves everything to JSON. The LLM call is wrapped in `try`/`except`, so this runs end to end even without Ollama; run it locally with Ollama for a real summary.

### Load the data

In [1]:
csv_text = """product,category,units,revenue
Widget,hardware,120,2400
Gadget,hardware,80,3200
Cable,accessory,300,1500
Case,accessory,150,1200
App,software,50,5000"""

import pandas as pd
from io import StringIO
df = pd.read_csv(StringIO(csv_text))
print(df)

  product   category  units  revenue
0  Widget   hardware    120     2400
1  Gadget   hardware     80     3200
2   Cable  accessory    300     1500
3    Case  accessory    150     1200
4     App   software     50     5000


### 1. Compute facts (Pandas)

In [2]:
def compute_facts(df) -> dict:
    return {
        "rows": int(len(df)),
        "total_revenue": int(df["revenue"].sum()),
        "top_category_by_revenue": df.groupby("category")["revenue"].sum().idxmax(),
        "avg_units": round(float(df["units"].mean()), 1),
    }

facts = compute_facts(df)
print(facts)

{'rows': 5, 'total_revenue': 13300, 'top_category_by_revenue': 'hardware', 'avg_units': 140.0}


### 2. Build a prompt from the facts

In [3]:
import json

def build_prompt(facts: dict) -> str:
    return ("Summarize these sales facts in two sentences for a business owner:\n"
            + json.dumps(facts, indent=2))

prompt = build_prompt(facts)
print(prompt)

Summarize these sales facts in two sentences for a business owner:
{
  "rows": 5,
  "total_revenue": 13300,
  "top_category_by_revenue": "hardware",
  "avg_units": 140.0
}


### 3. Ask the LLM (graceful fallback)

In [4]:
import requests

def ask_llm(prompt: str) -> str:
    try:
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": "llama3.2", "prompt": prompt, "stream": False},
            timeout=30,
        )
        return resp.json()["response"].strip()
    except Exception as e:
        return f"(LLM not reached: {type(e).__name__}. Run locally with Ollama for a real summary.)"

summary = ask_llm(prompt)
print(summary)

(LLM not reached: ConnectionError. Run locally with Ollama for a real summary.)


### 4. Save facts + summary to JSON

In [5]:
import os

class Report:
    def __init__(self, path: str):
        self.path = path

    def save(self, facts: dict, summary: str) -> None:
        with open(self.path, "w") as f:
            json.dump({"facts": facts, "summary": summary}, f, indent=2)

Report("report.json").save(facts, summary)

with open("report.json") as f:
    print(f.read())

os.remove("report.json")   # clean up the demo file

{
  "facts": {
    "rows": 5,
    "total_revenue": 13300,
    "top_category_by_revenue": "hardware",
    "avg_units": 140.0
  },
  "summary": "(LLM not reached: ConnectionError. Run locally with Ollama for a real summary.)"
}


Every Module 1 skill in one flow: functions, type hints, a class, files and JSON, Pandas, and an LLM call.